# 01 · msci-code-fast (milestone M1)

Inline code completion model. **Student** Qwen2.5-Coder-0.5B, **teacher** Qwen2.5-Coder-7B in 4-bit,
trained on fill-in-the-middle spans cut from pinned, permissively licensed Python repositories.

| Step | What happens | T4 time (est.) |
|---|---|---|
| prepare | clone repos, cut FIM spans, decontaminate, split by file | ~10 min |
| teacher_generate | teacher fills held-out holes (ceiling) + 25% of train holes | ~1–2 h |
| verify | keep teacher completions that parse and stay close to the original | ~5 min |
| teacher_logits | teacher's top-20 next-token probabilities on every target | ~30–60 min |
| train_sft | LoRA fine-tune on original + accepted teacher spans | ~20–30 min |
| train_kd | continue with CE + top-k KL against the teacher | ~20–30 min |
| evaluate | base vs SFT vs final vs teacher, latency, HumanEval+, gates | ~20–30 min |
| quantise | merge LoRA, export GGUF q8_0 for CPU serving | ~5 min |
| register | copy to the model store, write model card, update catalogue.yaml | <1 min |

Each step is resumable: if Colab disconnects, re-run the setup cell and the same step cell.
Use the timings from notebook 00 to plan sessions.

In [ ]:
# --- Setup: GPU check, Google Drive, repo, packages (run every session) ---
import glob, json, os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
SVLM_ROOT = '/content/drive/MyDrive/svlm'       # run outputs, checkpoints and the model store live here
REPO_ON_DRIVE = f'{SVLM_ROOT}/svlm-catalogue'   # copy the repo folder here once (Drive for Desktop works)
REPO = '/content/svlm-catalogue'                # working copy on Colab's fast local disk
os.environ['SVLM_ROOT'] = SVLM_ROOT
os.makedirs(SVLM_ROOT, exist_ok=True)

def _find_repo():
    """Return the first folder that looks like the repo (has configs/ and svlm/), searching
    Colab's local disk and Drive, including one extra nesting level from zip/folder uploads."""
    bases = ['/content/svlm-catalogue', REPO_ON_DRIVE, '/content/drive/MyDrive/svlm-catalogue', '/content/svlm', '/content']
    for b in bases:
        for cand in (b, os.path.join(b, 'svlm-catalogue')):
            if os.path.isdir(os.path.join(cand, 'configs')) and os.path.isdir(os.path.join(cand, 'svlm')):
                return cand
    return None

found = _find_repo()
if found is None:
    from google.colab import files
    print('Repo not found. Upload svlm-catalogue.zip (from C:\\projects\\TechFest):')
    up = files.upload()
    subprocess.run(['unzip', '-q', '-o', next(iter(up)), '-d', '/content'], check=True)
    found = _find_repo()
    assert found, 'zip did not contain the svlm-catalogue folder'
if found.startswith('/content/drive'):
    # work from a copy on Colab's local disk: faster, and Drive stays the clean source
    subprocess.run(['rm', '-rf', REPO], check=True)
    subprocess.run(['cp', '-r', found, REPO], check=True)
    found = REPO
REPO = found
print('repo:', REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)

locks = sorted(glob.glob(f'{SVLM_ROOT}/env/lock-colab-*.txt'))
REQ = locks[-1] if locks else 'requirements/colab.txt'
print('installing from', REQ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQ], check=True)
# Colab preinstalls an old torchao; recent peft refuses to build LoRA layers while it is present (unused here)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=False)

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} | compute {p.major}.{p.minor} | {p.total_memory/1e9:.1f} GB | bf16: {torch.cuda.is_bf16_supported(including_emulation=False)}')

## Configuration
Change values with `--set key=value` on any step, or edit `configs/code_fast.yaml`.

In [ ]:
from pathlib import Path
from svlm.utils import read_json, read_jsonl
RUN = Path(SVLM_ROOT) / 'runs' / 'msci-code-fast' / '0.1.0-t4'
print(open('configs/code_fast.yaml').read())
!python pipelines/run_local.py --config configs/code_fast.yaml --status

## 1 · prepare

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps prepare

In [ ]:
print(json.dumps(read_json(RUN / 'prepare' / 'manifest.json'), indent=2))
s = read_jsonl(RUN / 'prepare' / 'train.jsonl')[0]
print('--- example', s['span_type'], s['path'], '---')
print(s['prefix'][-400:] + '\033[92m' + s['middle'] + '\033[0m' + s['suffix'][:200])

## 2 · teacher_generate
The slowest step. Shards of 200 are saved to Drive as they finish.

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps teacher_generate

## 3 · verify

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps verify

In [ ]:
print(json.dumps(read_json(RUN / 'verify' / 'filter_report.json'), indent=2))

## 4 · teacher_logits

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps teacher_logits

## 5 · train_sft

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps train_sft

In [ ]:
import matplotlib.pyplot as plt
def plot(step, keys=('loss',)):
    h = read_json(RUN / step / 'train_log.json')['log_history']
    for k in keys:
        pts = [(r['step'], r[k]) for r in h if k in r]
        if pts: plt.plot(*zip(*pts), label=k)
    plt.xlabel('step'); plt.legend(); plt.title(step); plt.grid(alpha=.3); plt.show()
plot('train_sft')

## 6 · train_kd
Loss = 0.5 × cross-entropy + 0.5 × KL to the teacher's top-20 distribution.

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps train_kd

In [ ]:
plot('train_kd', keys=('loss', 'ce', 'kl'))

## 7 · evaluate

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps evaluate

In [ ]:
m = read_json(RUN / 'evaluate' / 'metrics.json')
rows = [(k, m[k]['exact_match'], m[k]['edit_similarity'], m[k]['parse_rate']) for k in ('teacher', 'base', 'sft', 'final') if m.get(k)]
print(f"{'model':8s} {'exact':>7s} {'edit-sim':>9s} {'parse':>7s}")
for r in rows: print(f'{r[0]:8s} {r[1]:7.3f} {r[2]:9.3f} {r[3]:7.3f}')
print('\nlatency:', m['final'].get('latency')); print('humaneval:', m.get('humaneval'))
print('\nGATES:', m['gates']['verdict'].upper())
for k, v in m['gates']['checks'].items(): print(f"  {'PASS' if v['pass'] else 'FAIL'}  {k}: {v['value']} (threshold {v.get('threshold', v.get('base'))})")

## 8 · quantise

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps quantise

## 9 · register
Fails on purpose if the gates failed. Fix the cause and re-run, or add `--set register.force=true` to store it as a non-live version.

In [ ]:
!python pipelines/run_local.py --config configs/code_fast.yaml --steps register

In [ ]:
from IPython.display import Markdown, display
display(Markdown((Path(SVLM_ROOT) / 'model_store' / 'msci-code-fast' / '0.1.0-t4' / 'model_card.md').read_text()))
print(open(Path(SVLM_ROOT) / 'model_store' / 'catalogue.yaml').read())

## 10 · Try it

Quick completion with the merged model. The gateway serves the GGUF file on CPU with llama.cpp in milestone M5.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from svlm.fim import fim_prompt, trim_completion
mdir = RUN / 'merged'
tok = AutoTokenizer.from_pretrained(mdir); model = AutoModelForCausalLM.from_pretrained(mdir, dtype=torch.float16).cuda()
prefix = "import numpy as np\n\ndef sharpe_ratio(returns, rf=0.0, periods=252):\n"
suffix = "\n    return mean / std * np.sqrt(periods)\n"
ids = tok(fim_prompt(prefix, suffix), return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=64, do_sample=False)
print(prefix + '\033[92m' + trim_completion(tok.decode(out[0][ids['input_ids'].shape[1]:]), 'block', suffix) + '\033[0m' + suffix)

## Get the model onto your machine

The release folder is `MyDrive/svlm/model_store/msci-code-fast/0.1.0-t4/` (GGUF, merged weights, model card,
metrics, lineage) plus `MyDrive/svlm/model_store/catalogue.yaml`. With Google Drive for Desktop it syncs to
your computer automatically; copy it into the gateway server's model store (for example
`C:\projects\TechFest\model_store\`).